In [ ]:
# IMPORTS BÁSICOS
import os, sys
import pandas as pd
import numpy as np
import time

# Frameworks de deep learning y NeuralForecast
import torch
import pytorch_lightning as pl
from neuralforecast import NeuralForecast, __version__ as nf_version
# Intentamos importar GRU del paquete; si no existe, sustituir por la clase equivalente
try:
    from neuralforecast.models import GRU
except Exception:
    # Si la librería no expone GRU así, dejar la importación para ajustar manualmente
    GRU = None
from neuralforecast.losses.pytorch import MAE

# Importar utilidades del repo (mismos helpers que en el cuaderno LSTM)
lib_dir = os.path.join(os.getcwd(), 'lib')
if lib_dir not in sys.path:
    sys.path.insert(0, lib_dir)

from lib.dl_utils import preparar_datos_neuralforecast, preparar_variables_estaticas
from lib.metricas import calcular_metricas, resumen_metricas
from lib.graficos_dl import grafico_prediccion_diaria_agregada, grafico_prediccion_por_cluster, grafico_productos_por_cluster, dashboard_metricas_dl

print(f"torch: {torch.__version__}")
print(f"pytorch_lightning: {pl.__version__}")
print(f"neuralforecast: {nf_version}")
print("GRU disponible en neuralforecast.models: ", GRU is not None)


In [ ]:
# LECTURA DE DATOS (usar los datos ya preparados en /workspaces/TFMDS/datos)
df_train_raw = pd.read_csv('datos/df_train_dl.csv', sep=';', parse_dates=['idSecuencia'])
df_test_raw = pd.read_csv('datos/df_test_dl.csv', sep=';', parse_dates=['idSecuencia'])
print(f"Datos cargados. Train: {df_train_raw.shape}, Test: {df_test_raw.shape}")

In [ ]:
# PREPARAR DATOS PARA NEURALFORECAST
df_train_nf, df_test_nf = preparar_datos_neuralforecast(df_train_raw, df_test_raw, col_fecha='idSecuencia', col_producto='producto', col_target='udsVenta')
# Preparar variables estáticas (si aplica)
df_train_nf, df_test_nf, static_df = preparar_variables_estaticas(df_train_nf, df_test_nf, stat_exog_list=['Cluster_0','Cluster_1','Cluster_2','Cluster_3'])
print(f"Formatos NF: train {df_train_nf.shape}, test {df_test_nf.shape}")

In [ ]:
# CONFIGURACIÓN DEL MODELO GRU (ajusta hiperparámetros según necesidades)
HORIZON = 30
if GRU is None:
    raise ImportError('GRU model not found in neuralforecast.models; adapta la importación antes de ejecutar.')
modelo_gru = GRU(
    h=HORIZON,
    input_size=60,
    loss=MAE(),
    max_steps=500,
    encoder_hidden_size=128,
    encoder_n_layers=2,
    decoder_hidden_size=128,
    decoder_layers=2,
    learning_rate=1e-3,
    scaler_type='standard',
    batch_size=32,
    random_seed=42,
    futr_exog_list=[ 'bolOpen', 'bolHoliday', 'bolPromocion', 'dia_semana_sin', 'dia_semana_cos', 'mes_sin', 'mes_cos' ],
    hist_exog_list=[ 'lag_ventas_1','lag_ventas_2','lag_ventas_3','lag_ventas_4' ],
    stat_exog_list=['Cluster_0','Cluster_1','Cluster_2','Cluster_3'],
    enable_progress_bar=True
)
print('Modelo GRU configurado: ', modelo_gru)

In [ ]:
# ENTRENAMIENTO
nf = NeuralForecast(models=[modelo_gru], freq='D')
print('Entrenando...')
start = time.perf_counter()
nf.fit(df=df_train_nf, static_df=static_df)
print(f'Entrenamiento completado en {time.perf_counter()-start:.2f}s')

In [ ]:
# PREDICCIÓN
y_hat = nf.predict(futr_df=df_test_nf)
print('Predicciones generadas. Shape:', getattr(y_hat, 'shape', None))

In [ ]:
# RECONSTRUIR Y CALCULAR MÉTRICAS (misma lógica que en LSTM)
if y_hat.index.name == 'unique_id':
    y_hat = y_hat.reset_index()
# rename if needed and merge with df_test_nf
if 'GRU' in y_hat.columns:
    y_hat = y_hat.rename(columns={'GRU':'prediccion'})
elif 'prediccion' not in y_hat.columns:
    pred_cols = [c for c in y_hat.columns if c not in ['unique_id','ds']]
    if pred_cols:
        y_hat = y_hat.rename(columns={pred_cols[0]:'prediccion'})
df_test_pred = df_test_nf[['unique_id','ds','y']].merge(y_hat[['unique_id','ds','prediccion']], on=['unique_id','ds'], how='left')
df_test_pred['idSecuencia']=df_test_pred['ds']
df_test_pred['producto']=df_test_pred['unique_id']
df_test_pred['udsVenta']=df_test_pred['y']
df_test_pred['prediccion'] = df_test_pred['prediccion'].clip(lower=0)
df_test_pred['error'] = df_test_pred['prediccion'] - df_test_pred['udsVenta']
df_test_pred['error_abs'] = np.abs(df_test_pred['error'])
print('Dataset de predicciones preparado:', df_test_pred.shape)

In [ ]:
# CALCULO DE METRICAS
df_valid = df_test_pred.dropna(subset=['prediccion','udsVenta'])
metricas_gru = calcular_metricas(y=df_valid['udsVenta'], y_pred=df_valid['prediccion'], name='GRU')
resumen_metricas([metricas_gru])
todas_metricas = [metricas_gru]